In [4]:
import gurobipy as gp
from gurobipy import GRB
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_TEST_GRAFICO"
d={}
B_num={}
p={1:869.434,2:1533.228,3:2430.901,4:3169.766,5:3947.629,6:4690.218,7:5514.84,8:6051.184,9:6693.546,
   10:7264.281,11:8095.383,12:8530.091,13:9088.648,14:9600.03,15:10486.354,16:11616.937,17:12249.156,
   18:12605.908,19:14312.13,20:15063.13,21:16036.058,22:16509.946,23:17643.719,24:18142.476,25:19010.222}
idx_p=gp.tuplelist(p)
maxi=0


for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']<=3.03]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            d[(j+1, k)] = row['dist_Acum']
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
print(B_max)

print(M)
idx_d=gp.tuplelist(d)
#print(d)
#print(p)
#print(B[1])
print(B_num)
idx_B=gp.tuplelist(B_num)
print(idx_B)
#print(idx_d)
S=sum(B_num.values())
print(S)

1098
38036.11949999995
{1: 951, 2: 1098, 3: 870, 4: 830, 5: 699}
[1, 2, 3, 4, 5]
4448


In [8]:
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=25*0.8 for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 226978 rows, 115653 columns and 1000800 nonzeros
Model fingerprint: 0x7c3757a2
Variable types: 4453 continuous, 111200 integer (111200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Presolve removed 210985 rows and 105609 columns
Presolve time: 0.72s
Presolved: 15993 rows, 10044 columns, 63758 nonzeros
Variable types: 2424 continuous, 7620 integer (7620 binary)
Starting NoRel heuristic
Found phase-1 solution: relaxation 511.722
Found phase-1 solution: relaxation 501.353
Found phase-1 solution: relaxation 2
Elapsed time for NoRel 

In [5]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 15 para la parada 1.
Base de datos 2 punto elegido 12 para la parada 1.
Base de datos 4 punto elegido 1 para la parada 1.
Base de datos 5 punto elegido 1 para la parada 1.
Base de datos 1 punto elegido 26 para la parada 2.
Base de datos 2 punto elegido 29 para la parada 2.
Base de datos 3 punto elegido 3 para la parada 2.
Base de datos 4 punto elegido 2 para la parada 2.
Base de datos 5 punto elegido 8 para la parada 2.
Base de datos 1 punto elegido 40 para la parada 3.
Base de datos 2 punto elegido 48 para la parada 3.
Base de datos 3 punto elegido 13 para la parada 3.
Base de datos 4 punto elegido 13 para la parada 3.
Base de datos 5 punto elegido 12 para la parada 3.
Base de datos 1 punto elegido 49 para la parada 4.
Base de datos 2 punto elegido 58 para la parada 4.
Base de datos 3 punto elegido 35 para la parada 4.
Base de datos 4 punto elegido 24 para la parada 4.
Base de datos 5 punto elegido 14 para la parada 4.
Base de datos 1 punto elegido 67 par

In [7]:
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.reset()
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Discarded solution information
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 226978 rows, 115653 columns and 1000800 nonzeros
Model fingerprint: 0x6e6ec0d7
Variable types: 4453 continuous, 111200 integer (111200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+03]
  RHS range        [1e+00, 8e+04]
Found heuristic solution: objective 1314387.6269
Presolve removed 106389 rows and 0 columns (presolve time = 22s) ...
Presolve removed 106389 rows and 0 columns
Presolve time: 21.87s
Presolved: 120589 rows, 115653 columns, 681633 nonzeros
Variable types: 4453 continuous, 111200 integer (111200 binary)
Starting NoRel heur

In [9]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 26 para la parada 1.
Base de datos 2 punto elegido 29 para la parada 1.
Base de datos 3 punto elegido 7 para la parada 1.
Base de datos 4 punto elegido 11 para la parada 1.
Base de datos 5 punto elegido 8 para la parada 1.
Base de datos 1 punto elegido 40 para la parada 2.
Base de datos 2 punto elegido 48 para la parada 2.
Base de datos 3 punto elegido 13 para la parada 2.
Base de datos 4 punto elegido 13 para la parada 2.
Base de datos 5 punto elegido 11 para la parada 2.
Base de datos 1 punto elegido 47 para la parada 3.
Base de datos 2 punto elegido 58 para la parada 3.
Base de datos 3 punto elegido 35 para la parada 3.
Base de datos 4 punto elegido 28 para la parada 3.
Base de datos 5 punto elegido 14 para la parada 3.
Base de datos 1 punto elegido 61 para la parada 4.
Base de datos 2 punto elegido 80 para la parada 4.
Base de datos 3 punto elegido 63 para la parada 4.
Base de datos 4 punto elegido 40 para la parada 4.
Base de datos 5 punto elegido 30 

In [11]:
m.Params.TimeLimit = 1200
m.Params.NoRelHeurTime = 600
m.reset()
m.optimize()

Set parameter TimeLimit to value 1200
Discarded solution information
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 226978 rows, 115653 columns and 1000800 nonzeros
Model fingerprint: 0x6e6ec0d7
Variable types: 4453 continuous, 111200 integer (111200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+03]
  RHS range        [1e+00, 8e+04]
Found heuristic solution: objective 1314387.6269
Presolve removed 106389 rows and 0 columns (presolve time = 36s) ...
Presolve removed 106389 rows and 0 columns
Presolve time: 35.76s
Presolved: 120589 rows, 115653 columns, 681633 nonzeros
Variable types: 4453 continuous, 111200 integer (111200 binary)
Starting NoRel heuristic
Found heuristic solution: objective

In [15]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento de {i} : {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 26 para la parada 1.
Base de datos 2 punto elegido 29 para la parada 1.
Base de datos 3 punto elegido 7 para la parada 1.
Base de datos 4 punto elegido 11 para la parada 1.
Base de datos 5 punto elegido 8 para la parada 1.
Base de datos 1 punto elegido 40 para la parada 2.
Base de datos 2 punto elegido 48 para la parada 2.
Base de datos 3 punto elegido 13 para la parada 2.
Base de datos 4 punto elegido 13 para la parada 2.
Base de datos 5 punto elegido 11 para la parada 2.
Base de datos 1 punto elegido 47 para la parada 3.
Base de datos 2 punto elegido 58 para la parada 3.
Base de datos 3 punto elegido 35 para la parada 3.
Base de datos 4 punto elegido 28 para la parada 3.
Base de datos 5 punto elegido 14 para la parada 3.
Base de datos 1 punto elegido 61 para la parada 4.
Base de datos 2 punto elegido 80 para la parada 4.
Base de datos 3 punto elegido 63 para la parada 4.
Base de datos 4 punto elegido 40 para la parada 4.
Base de datos 5 punto elegido 30 